# 📓 GOOGLE COLAB NOTEBOOK

## JSON → Corporate PPT Generator

## 🟦 CELL 1 — Install Dependencies

In [ ]:
!pip install python-pptx pydantic

## 🟦 CELL 2 — Imports & Constants

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.chart.data import CategoryChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_LEGEND_POSITION
import json

## 🟦 CELL 3 — Theme Definition (LOCKED)

> Single theme used for **all slides**

In [ ]:
THEME = {
    "background": RGBColor(245, 245, 245),
    "title_color": RGBColor(47, 93, 140),
    "text_color": RGBColor(31, 41, 51),
    "muted_text": RGBColor(95, 108, 114),
    "table_header_bg": RGBColor(238, 242, 246),
    "border": RGBColor(217, 221, 225),

    "status": {
        "completed": RGBColor(76, 175, 80),
        "in-progress": RGBColor(74, 144, 226),
        "pending": RGBColor(224, 168, 0)
    },

    "font": {
        "family": "Calibri",
        "title": 32,
        "section": 22,
        "body": 14,
        "small": 11
    }
}

## 🟦 CELL 4 — PPT Helper Functions

In [ ]:
def add_cover_slide(slide, title, subtitle):
    # Background Image/Shape for visual interest
    bg_shape = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), Inches(13.33), Inches(7.5)
    )
    bg_shape.fill.solid()
    bg_shape.fill.fore_color.rgb = THEME["title_color"]
    
    # White overlay for text
    overlay = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(1), Inches(1.5), Inches(11.33), Inches(4.5)
    )
    overlay.fill.solid()
    overlay.fill.fore_color.rgb = RGBColor(255, 255, 255)
    overlay.line.color.rgb = RGBColor(255, 255, 255)

    # Centered Title
    title_box = slide.shapes.add_textbox(Inches(1.5), Inches(2.5), Inches(10.33), Inches(2))
    tf = title_box.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = title
    p.font.size = Pt(54)
    p.font.bold = True
    p.font.color.rgb = THEME["title_color"]
    p.alignment = PP_ALIGN.CENTER

    # Centered Subtitle
    subtitle_box = slide.shapes.add_textbox(Inches(1.5), Inches(4.5), Inches(10.33), Inches(1))
    tf = subtitle_box.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = subtitle
    p.font.size = Pt(28)
    p.font.color.rgb = THEME["muted_text"]
    p.alignment = PP_ALIGN.CENTER

def add_title(slide, text):
    # Create a title text box
    title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12.33), Inches(1))
    tf = title_shape.text_frame
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(36)
    p.font.bold = True
    p.font.color.rgb = THEME["title_color"]
    
    # Add a decorative underline
    line = slide.shapes.add_shape(
        MSO_SHAPE.RECTANGLE, Inches(0.5), Inches(1.1), Inches(12.33), Inches(0.05)
    )
    line.fill.solid()
    line.fill.fore_color.rgb = THEME["status"]["in-progress"] # Accent color
    line.line.fill.background()

def add_text(slide, text, top=1.5):
    box = slide.shapes.add_textbox(Inches(0.5), Inches(top), Inches(12.33), Inches(5))
    tf = box.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(20) # Larger body text
    p.font.color.rgb = THEME["text_color"]

def add_banner(slide, text, top=1.5):
    box = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(0.5), Inches(top), Inches(12.33), Inches(1.5)
    )
    box.fill.solid()
    box.fill.fore_color.rgb = RGBColor(235, 242, 250) # Light blue bg
    box.line.color.rgb = THEME["title_color"]
    
    tf = box.text_frame
    tf.margin_left = Inches(0.3)
    tf.margin_right = Inches(0.3)
    tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    
    p = tf.paragraphs[0]
    p.text = text
    p.font.size = Pt(20)
    p.font.color.rgb = THEME["text_color"]
    p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 5 — Table Renderer

In [ ]:
def add_table(slide, table_json, top=2.0):
    """
    Renders a table with a clean, professional white look.
    """
    rows = len(table_json["rows"]) + 1
    cols = len(table_json["columns"])
    
    # Center the table
    table_width = Inches(12.33)
    left = Inches(0.5)
    
    shape = slide.shapes.add_table(
        rows, cols,
        left, Inches(top),
        table_width, Inches(0.6 * rows)
    )
    table = shape.table

    # 1. Header Styling
    for i, col in enumerate(table_json["columns"]):
        cell = table.cell(0, i)
        cell.text = col["label"]
        cell.fill.solid()
        # Dark Blue Header
        cell.fill.fore_color.rgb = THEME["title_color"] 
        
        p = cell.text_frame.paragraphs[0]
        p.font.bold = True
        p.font.size = Pt(14)
        p.font.color.rgb = RGBColor(255, 255, 255)
        p.alignment = PP_ALIGN.CENTER
        cell.vertical_anchor = MSO_ANCHOR.MIDDLE

    # 2. Row Styling
    for r, row in enumerate(table_json["rows"], start=1):
        for c, col in enumerate(table_json["columns"]):
            val = str(row.get(col["key"], ""))
            cell = table.cell(r, c)
            cell.text = val
            
            # Clean White Background for all rows (Professional Look)
            cell.fill.solid()
            cell.fill.fore_color.rgb = RGBColor(255, 255, 255)
            
            # Add a bottom border to separate rows (Visual trick using transparency or just rely on grid)
            # PPTX tables have borders by default, we just ensure text is readable.
            
            p = cell.text_frame.paragraphs[0]
            p.font.size = Pt(12)
            p.font.color.rgb = THEME["text_color"]
            p.alignment = PP_ALIGN.LEFT
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE
            
            # Center align status or short columns
            if len(val) < 15:
                p.alignment = PP_ALIGN.CENTER

            # Status Color Coding (Text Color)
            if col["key"] == "status":
                status_key = val.lower().replace(" ", "-")
                if status_key in THEME["status"]:
                    p.font.color.rgb = THEME["status"][status_key]
                    p.font.bold = True

def add_image_slide(slide, image_path, title, caption=""):
    """
    Adds an image to the slide. Handles missing files gracefully.
    """
    # Title is already added by main loop
    
    if not os.path.exists(image_path):
        # Placeholder if image missing
        box = slide.shapes.add_textbox(Inches(1), Inches(2.5), Inches(11), Inches(4))
        box.text_frame.text = f"[IMAGE NOT FOUND: {image_path}]\nPlease check the path in JSON."
        return

    # Add Image (Centered, max height 4.5 inches)
    pic = slide.shapes.add_picture(image_path, Inches(1), Inches(2.0), height=Inches(4.5))
    
    # Center the image horizontally
    # (slide_width - pic_width) / 2
    pic.left = int((Inches(13.333) - pic.width) / 2)
    
    if caption:
        box = slide.shapes.add_textbox(Inches(1), Inches(6.6), Inches(11.33), Inches(0.5))
        p = box.text_frame.paragraphs[0]
        p.text = caption
        p.font.size = Pt(11)
        p.font.italic = True
        p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 6 — Card Renderer

In [ ]:
def add_cards_slide(slide, cards_data, top=2.0):
    """
    Dynamically arranges cards in a grid.
    Supports 2, 3, or 4 cards automatically.
    """
    count = len(cards_data)
    if count == 0: return

    # Layout Logic
    slide_width = 13.33
    margin = 0.5
    spacing = 0.3
    
    available_width = slide_width - (2 * margin) - ((count - 1) * spacing)
    card_width = available_width / count
    card_height = 4.5

    for i, card in enumerate(cards_data):
        left = margin + (i * (card_width + spacing))
        
        # 1. Card Box (White with Shadow/Border)
        box = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(left), Inches(top), Inches(card_width), Inches(card_height)
        )
        box.fill.solid()
        box.fill.fore_color.rgb = RGBColor(255, 255, 255)
        box.line.color.rgb = THEME["border"]
        box.line.width = Pt(1)
        
        # 2. Header Strip (Colored top part of card)
        header = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE, # Top rounded only isn't easy, so we overlay
            Inches(left), Inches(top), Inches(card_width), Inches(0.8)
        )
        header.fill.solid()
        header.fill.fore_color.rgb = THEME["table_header_bg"]
        header.line.fill.background()
        
        # 3. Title Text
        tf = header.text_frame
        p = tf.paragraphs[0]
        p.text = card["title"]
        p.font.bold = True
        p.font.size = Pt(16)
        p.font.color.rgb = THEME["title_color"]
        p.alignment = PP_ALIGN.CENTER
        
        # 4. Content Text (Bullet points)
        # We create a new text box ON TOP of the white card body to handle margins better
        content_box = slide.shapes.add_textbox(
            Inches(left + 0.1), Inches(top + 0.9), Inches(card_width - 0.2), Inches(card_height - 1.0)
        )
        tf = content_box.text_frame
        tf.word_wrap = True
        
        for item in card["items"]:
            p = tf.add_paragraph()
            p.text = f"• {item}"
            p.font.size = Pt(14)
            p.font.color.rgb = THEME["text_color"]
            p.space_after = Pt(6)

## 🟦 CELL 7 — Timeline Renderer

In [ ]:
def add_gantt_chart(slide, gantt_data):
    """
    Renders a professional Gantt Chart.
    """
    tasks = gantt_data["tasks"]
    if not tasks: return

    # Config
    start_x = 3.5  # Space for Task Names
    start_y = 2.5
    row_height = 0.6
    chart_width = 9.0
    
    # Find timeline range
    min_week = min(t["start_week"] for t in tasks)
    max_week = max(t["start_week"] + t["duration"] for t in tasks)
    total_weeks = max_week - min_week + 1
    
    week_width = chart_width / total_weeks

    # 1. Draw Header (Weeks)
    for w in range(total_weeks):
        x = start_x + (w * week_width)
        
        # Grid line
        line = slide.shapes.add_shape(
            MSO_SHAPE.LINE_INVERSE, Inches(x), Inches(start_y), Inches(0), Inches(len(tasks) * row_height + 0.5)
        )
        line.line.color.rgb = THEME["border"]
        line.line.dash_style = 1 # Dash
        
        # Label
        lbl = slide.shapes.add_textbox(Inches(x), Inches(start_y - 0.4), Inches(week_width), Inches(0.4))
        p = lbl.text_frame.paragraphs[0]
        p.text = f"W{min_week + w}"
        p.font.size = Pt(10)
        p.font.bold = True
        p.alignment = PP_ALIGN.CENTER

    # 2. Draw Tasks
    for i, task in enumerate(tasks):
        y = start_y + (i * row_height)
        
        # Task Name (Left Side)
        name_box = slide.shapes.add_textbox(Inches(0.5), Inches(y), Inches(2.8), Inches(row_height))
        p = name_box.text_frame.paragraphs[0]
        p.text = task["name"]
        p.font.size = Pt(12)
        p.font.bold = True
        p.alignment = PP_ALIGN.RIGHT
        
        # Task Bar
        bar_start = start_x + ((task["start_week"] - min_week) * week_width)
        bar_width = task["duration"] * week_width
        
        bar = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(bar_start), Inches(y + 0.1), Inches(bar_width), Inches(row_height - 0.2)
        )
        bar.fill.solid()
        
        # Color logic
        if "Design" in task["name"]: color = THEME["status"]["in-progress"]
        elif "Test" in task["name"]: color = THEME["status"]["pending"]
        else: color = THEME["title_color"]
        
        bar.fill.fore_color.rgb = color
        bar.line.color.rgb = color
        
        # Progress % Label inside bar
        if "progress" in task:
            p = bar.text_frame.paragraphs[0]
            p.text = f"{task['progress']}%"
            p.font.size = Pt(10)
            p.font.color.rgb = RGBColor(255, 255, 255)
            p.alignment = PP_ALIGN.CENTER

## 🟦 CELL 8 — FULL JSON INPUT (Random Project Proposal)

> **ONLY this JSON controls the PPT**

In [ ]:
# ==========================================
# 🟦 MASTER JSON CONFIGURATION
# ==========================================
# This JSON controls the entire presentation.
# You can pass this structure to an AI to generate new content.
#
# TYPES EXPLAINED:
# - "cover": Title slide. Needs 'title', 'subtitle'.
# - "content": Standard text slide. Needs 'title', 'banner' (highlight box), 'text'.
# - "table": Data table. Needs 'columns' (key/label) and 'rows' (data objects).
# - "cards": Grid of info cards. Needs 'cards' list (title, items).
# - "gantt": Project schedule. Needs 'tasks' (name, start_week, duration, progress).
# - "chart": Bar chart. Needs 'categories' (x-axis), 'values' (y-axis), 'series_name'.
# - "image": Full slide image. Needs 'image_path', 'caption'.

ppt_json = {
  "meta": {
    "title": "DriveNow: Car Rental Platform",
    "author": "Ashish Jha",
    "date": "Dec 2025"
  },
  "slides": [
    # SLIDE 1: COVER
    {
      "type": "cover",
      "title": "DriveNow Car Rental",
      "subtitle": "Digital Transformation Proposal"
    },

    # SLIDE 2: EXECUTIVE SUMMARY (Content)
    {
      "type": "content",
      "title": "Executive Summary",
      "banner": "A comprehensive solution to digitize fleet management and customer bookings.",
      "text": "We propose a cloud-native platform that enables real-time inventory tracking, seamless mobile bookings for customers, and predictive maintenance for the fleet. This will reduce operational costs by 20% and increase utilization by 15%."
    },

    # SLIDE 3: MARKET GROWTH (Chart)
    {
      "type": "chart",
      "title": "Projected Market Growth",
      "chart": {
        "categories": ["2025", "2026", "2027", "2028"],
        "series_name": "Revenue ($M)",
        "values": [1.2, 2.5, 4.8, 8.0]
      }
    },

    # SLIDE 4: CORE MODULES (Cards - Dynamic Grid)
    {
      "type": "cards",
      "title": "Core Modules",
      "cards": [
        {
          "title": "Customer App",
          "items": ["Instant Booking", "Digital Key Access", "Loyalty Rewards", "24/7 Support Chat"]
        },
        {
          "title": "Admin Dashboard",
          "items": ["Fleet Tracking (GPS)", "Dynamic Pricing", "Maintenance Alerts", "Driver Verification"]
        },
        {
          "title": "Operations",
          "items": ["Car Cleaning Log", "Fuel Tracking", "Damage Inspection", "Staff Scheduling"]
        }
      ]
    },

    # SLIDE 5: PROJECT SCHEDULE (Gantt Chart)
    {
      "type": "gantt",
      "title": "Implementation Schedule",
      "gantt": {
        "tasks": [
          {"name": "Requirement Gathering", "start_week": 1, "duration": 2, "progress": 100},
          {"name": "UI/UX Design", "start_week": 2, "duration": 3, "progress": 80},
          {"name": "Backend Development", "start_week": 4, "duration": 6, "progress": 40},
          {"name": "Mobile App Dev", "start_week": 5, "duration": 5, "progress": 20},
          {"name": "Integration Testing", "start_week": 9, "duration": 2, "progress": 0},
          {"name": "UAT & Launch", "start_week": 11, "duration": 2, "progress": 0}
        ]
      }
    },

    # SLIDE 6: BUDGET (Table)
    {
      "type": "table",
      "title": "Budget Estimation",
      "table": {
        "columns": [
          {"key": "item", "label": "Cost Item"},
          {"key": "desc", "label": "Description"},
          {"key": "cost", "label": "Cost (USD)"},
          {"key": "status", "label": "Approval"}
        ],
        "rows": [
          {"item": "Software Dev", "desc": "Frontend, Backend, Mobile Apps", "cost": "$45,000", "status": "Approved"},
          {"item": "Infrastructure", "desc": "AWS Cloud, Database, CDN", "cost": "$5,000", "status": "Pending"},
          {"item": "Licensing", "desc": "Maps API, SMS Gateway", "cost": "$2,000", "status": "Approved"},
          {"item": "Marketing", "desc": "Launch Campaign & SEO", "cost": "$8,000", "status": "Pending"},
          {"item": "Total", "desc": "Estimated Project Cost", "cost": "$60,000", "status": ""}
        ]
      }
    },
    
    # SLIDE 7: ARCHITECTURE (Image Placeholder)
    {
      "type": "image",
      "title": "System Architecture",
      "image_path": "architecture_diagram.png", 
      "caption": "Figure 1: High-level cloud architecture diagram showing microservices."
    }
  ]
}

## 🟦 CELL 9 — PPT Generator (JSON → PPT)

In [ ]:
prs = Presentation()
prs.slide_width = Inches(13.333)
prs.slide_height = Inches(7.5)

for slide_json in ppt_json["slides"]:
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    
    # Background
    background = slide.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = THEME["background"]

    # 1. Cover Slide
    if slide_json["type"] == "cover":
        add_cover_slide(slide, slide_json["title"], slide_json.get("subtitle", ""))
        continue

    # 2. Standard Title for all other slides
    add_title(slide, slide_json["title"])

    # 3. Content Renderers
    if slide_json["type"] == "content":
        add_banner(slide, slide_json["banner"], top=1.8)
        add_text(slide, slide_json["text"], top=3.8)

    elif slide_json["type"] == "table":
        add_table(slide, slide_json["table"], top=2.0)

    elif slide_json["type"] == "chart":
        add_chart(slide, slide_json["chart"], top=2.0)

    elif slide_json["type"] == "cards":
        # Pass the whole list of cards to the dynamic renderer
        add_cards_slide(slide, slide_json["cards"], top=2.0)

    elif slide_json["type"] == "gantt":
        add_gantt_chart(slide, slide_json["gantt"])
        
    elif slide_json["type"] == "image":
        add_image_slide(slide, slide_json["image_path"], slide_json["title"], slide_json.get("caption", ""))

    # 4. Footer
    footer = slide.shapes.add_textbox(Inches(0.5), Inches(7.0), Inches(12.33), Inches(0.5))
    p = footer.text_frame.paragraphs[0]
    p.text = f"{ppt_json['meta']['title']} | {ppt_json['meta']['date']}"
    p.font.size = Pt(10)
    p.font.color.rgb = THEME["muted_text"]
    p.alignment = PP_ALIGN.RIGHT

## 🟦 CELL 10 — Save & Download PPT

In [ ]:
import base64
import os
from IPython.display import HTML, display

output_path = "project_proposal.pptx"
prs.save(output_path)
print(f"Presentation saved to: {output_path}")

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    
    # Create a clickable download link
    download_link = f'<a href="data:application/vnd.openxmlformats-officedocument.presentationml.presentation;base64,{b64}" download="{output_path}" style="font-size: 20px; font-weight: bold; color: blue;">⬇️ Click Here to Download PPT</a>'
    display(HTML(download_link))